# 06 - LLM Evaluation (No RAG)

Evaluates the Gemini-backed LLM classifier on the same frozen `test.csv` used by BERT, using
a versioned classification prompt (`prompts/classification/v1.yaml`) that asks only for one
of the four agency labels -- no `OUT_OF_SCOPE`/`UNCERTAIN`, no explanations, no self-reported
confidence in this MVP.

**Validation-only side pass (docs/BLUEPRINT.md Section 4):** before touching `test.csv`,
this notebook also runs the same classifier over the *validation* split. That pass exists
only to produce a validation macro F1 comparable to BERT's, for the demo's default-routing-
method decision (Section 10) -- it never influences or precedes the frozen test evaluation
below, and it is saved separately (`method=llm, split=validation`).

All logic lives in `src/newstart_ai/models/llm/`; this notebook only calls it, evaluates, and
saves results.

### Load the frozen split and set up the Gemini provider

**Purpose:** Load all three splits, then construct the `GeminiProvider` (this project's only
`LLMProvider` implementation) and load the versioned classification prompt it will use for
every request in this notebook.

**Why this step is necessary:** `GeminiProvider` reads its provider name, model name, and API
key location entirely from `configs/llm.yaml` and `.env` -- nothing about which LLM or which
model is hard-coded in this notebook or in `src/newstart_ai/models/llm/`. Loading the prompt
from `prompts/classification/v1.yaml` (rather than writing prompt text directly in this
notebook) means the exact wording sent to Gemini is version-controlled and reusable by the
RAG-enhanced classifier in notebook 08.

**Inputs:** `data/splits/*.csv`, `configs/llm.yaml`, `.env` (for the API key), and
`prompts/classification/v1.yaml`.

**Output:** `train_df`/`val_df`/`test_df`/`manifest`, `provider` (a ready-to-use
`GeminiProvider`), and `prompt` (the loaded, versioned prompt template).

**How to interpret the result:** The printed line confirms which provider, model, and
prompt version this run actually used -- important because all three are configurable and
could change between runs.

In [1]:
import sys
import time
from pathlib import Path

sys.path.insert(0, str(Path("..") / "src"))

from newstart_ai.config import load_settings
from newstart_ai.data import load_split
from newstart_ai.models.llm import GeminiProvider, load_classification_prompt
from newstart_ai.evaluation import evaluate_predictions, save_predictions, save_metrics_report
from newstart_ai.schemas import ClassificationResult

settings = load_settings()
train_df, val_df, test_df, manifest = load_split(settings)
ds_cfg = settings.base.dataset

# GeminiProvider reads provider/model/API key entirely from configs/llm.yaml + .env --
# swapping to a different provider or model later is a config change, not a code change.
provider = GeminiProvider(settings)
# The classification prompt is versioned and stored outside this notebook (prompts/), so
# its exact wording is reviewable and reusable by the RAG-enhanced classifier in notebook 08.
prompt = load_classification_prompt(settings)
print(f"provider: {settings.llm.provider}  model: {settings.llm.model}  prompt: {prompt.version}")

provider: gemini  model: gemini-3.6-flash  prompt: v1


### Define a reusable "classify every row" helper

**Purpose:** Define `classify_split()`, a small helper that sends every document in a given
DataFrame to Gemini and collects the results -- used identically for both the validation
pass below and the frozen test pass later in this notebook.

**Why this step is necessary:** The validation pass and the test pass need to do exactly
the same thing (classify every row, one at a time, recording successes and failures
separately) -- defining this once avoids duplicating that logic and guarantees both passes
behave identically. Handling a failed API call by recording it in `errors` (rather than
letting the whole notebook crash, or silently inventing a label) matches this project's rule
that every one of the four labels must be a real model decision, never a fallback.

**Inputs:** A DataFrame (`val_df` or `test_df`), the split's name (for bookkeeping), and the
already-created `provider`/`prompt` from the previous cell.

**Output:** A function object, `classify_split` -- it produces no output itself until it's
called below.

**How to interpret the result:** Nothing to interpret yet; this cell just defines the tool
used by the next several cells.

In [2]:
def classify_split(df, split_name: str, method: str = "llm"):
    """Classifies every row in df, returning (results, errors). A failed call is recorded
    in `errors` (document_id + error message) and excluded from results/metrics rather than
    silently assigned a fake label."""
    results = []
    errors = []
    texts = df[ds_cfg.text_column].tolist()
    doc_ids = df[ds_cfg.id_column].astype(str).tolist()
    true_labels = df[ds_cfg.label_column].tolist()

    for doc_id, text, true_label in zip(doc_ids, texts, true_labels):
        try:
            # provider.classify() sends the document text through the versioned prompt and
            # parses Gemini's structured JSON response into one of the four allowed labels.
            result = provider.classify(
                text,
                document_id=doc_id,
                prompt=prompt,
                method=method,
                extra_metadata={"split": split_name},
            )
            # true_label isn't known to the provider (it only sees the document text), so
            # it's attached here, after the fact, purely for scoring.
            result.true_label = true_label
            results.append(result)
        except Exception as exc:
            # A failed call (network error, unexpected response, etc.) is recorded and
            # skipped -- it is never given a made-up label just to keep a row count intact.
            errors.append({"document_id": doc_id, "error": str(exc)})

    print(f"{split_name}: {len(results)} succeeded, {len(errors)} failed")
    return results, errors

## Validation-only side pass

Used only to select the demo's default routing method (Section 10) -- not part of the
primary research comparison.

### Validation-only pass: classify the validation split

**Purpose:** Run every validation-set document through Gemini using the helper above.

**Why this step is necessary:** This pass exists *only* to produce a validation macro F1 for
the LLM that's comparable to BERT's validation macro F1 from notebook 04 -- it is what lets
notebook 08 later decide, fairly, which method should be the demo's default routing method.
It is completely separate from, and happens before, the frozen test-set pass further down --
the validation results here never influence or get mixed into the test-set numbers.

**Inputs:** `val_df` (121 documents).

**Output:** `val_results` (successful classifications) and `val_errors` (any failed calls,
displayed below so failures are visible rather than silent).

**How to interpret the result:** Ideally `val_errors` is empty (as it was in this run) --
any entries there would need to be investigated before trusting the validation macro F1
computed in the next cell.

In [3]:
val_results, val_errors = classify_split(val_df, split_name="validation")
val_errors

validation: 121 succeeded, 0 failed


[]

### Score the validation-only pass

**Purpose:** Turn `val_results` into a `MetricsReport` (accuracy, macro F1, per-class
metrics, latency, token usage, and cost), save it, and print the headline validation macro
F1 number.

**Why this step is necessary:** Saving this report (tagged `split="validation"`) keeps it
clearly separate on disk from the frozen test report saved later -- notebook 08 reads this
exact file back to compare against BERT's and LLM+RAG's validation scores.

**Inputs:** `val_results` (true/predicted labels, latencies, token usage, cost per
document).

**Output:** `val_report`, saved to `artifacts/reports/llm_validation_metrics.json`; the
predictions themselves are saved to `artifacts/predictions/llm_validation.json`.

**How to interpret the result:** The printed validation macro F1 is not this project's
headline LLM result (that's the test-set number below) -- it exists purely to support the
demo-routing-method decision made in notebook 08.

In [4]:
val_report = evaluate_predictions(
    true_labels=[r.true_label for r in val_results],
    predicted_labels=[r.predicted_label for r in val_results],
    label_order=settings.base.labels,
    method="llm",
    split="validation",
    latencies_ms=[r.latency_ms for r in val_results],
    # total_token_usage/total_estimated_cost let this method's real API cost be compared
    # against BERT's (which has no per-call cost once trained) later in notebook 09.
    total_token_usage=sum(r.token_usage.total_tokens for r in val_results if r.token_usage),
    total_estimated_cost=sum(r.estimated_cost for r in val_results if r.estimated_cost is not None),
)
save_predictions(val_results, method="llm", split="validation", settings=settings)
save_metrics_report(val_report, settings)
print(f"LLM validation macro F1: {val_report.macro_f1:.4f}")

LLM validation macro F1: 1.0000


## Frozen test-set evaluation

`test.csv` is touched here, exactly once, for the LLM method.

### Frozen test-set pass: classify the test split (touched here, exactly once)

**Purpose:** Run every test-set document through Gemini, using the exact same
`classify_split()` helper and the exact same versioned prompt as the validation pass above.

**Why this step is necessary:** This is the LLM method's one and only frozen evaluation --
the number this project reports as the LLM's real performance. Using the identical prompt
and code path as the validation pass (rather than a special "final" version) is what makes
the validation-to-test comparison meaningful in the first place.

**Inputs:** `test_df` (151 documents).

**Output:** `test_results` and `test_errors` (displayed below).

**How to interpret the result:** As with the validation pass, an empty `test_errors` list
means every document received a valid prediction; any errors here would need to be resolved
before the metrics below could be trusted.

In [5]:
test_results, test_errors = classify_split(test_df, split_name="test")
test_errors

test: 151 succeeded, 0 failed


[]

### Save row-level test predictions before scoring

**Purpose:** Write every individual test-document prediction to
`artifacts/predictions/llm_test.json`, before any summary metric is computed.

**Why this step is necessary:** Exactly as in the BERT evaluation notebook, saving raw
predictions first means the underlying evidence for every later metric (and the error
analysis in notebook 09) is preserved on disk, independent of whatever summary statistics
get computed from it.

**Inputs:** `test_results`.

**Output:** A JSON file on disk.

**How to interpret the result:** No output is printed; this is a pure side effect the next
cells and later notebooks depend on.

In [6]:
save_predictions(test_results, method="llm", split="test", settings=settings)

WindowsPath('D:/USD/Projects/a590/newstart-ai/newstart_ai_benchmark/artifacts/predictions/llm_test.json')

### Score the frozen test-set pass

**Purpose:** Compute the LLM's headline metrics for this project -- accuracy, macro/weighted
F1, per-class precision/recall/F1, confusion matrix, latency, token usage, and cost -- from
the 151 test-set predictions.

**Why this step is necessary:** This is the number that gets compared directly against
BERT's test macro F1 (notebook 05) and LLM+RAG's test macro F1 (notebook 08) in the final
comparison (notebook 09). Attaching the IRS small-sample note here means the caveat travels
with the report file itself.

**Inputs:** `test_results` (true/predicted labels, latencies, token usage, cost).

**Output:** `test_report`, saved to `artifacts/reports/llm_test_metrics.json`.

**How to interpret the result:** `test_report.macro_f1` is the LLM's headline result. The
token-usage and cost fields are what make it possible to weigh "is the extra cost of an LLM
call worth it compared to a free, already-trained BERT model?" in the final comparison.

In [7]:
test_report = evaluate_predictions(
    true_labels=[r.true_label for r in test_results],
    predicted_labels=[r.predicted_label for r in test_results],
    label_order=settings.base.labels,
    method="llm",
    split="test",
    latencies_ms=[r.latency_ms for r in test_results],
    total_token_usage=sum(r.token_usage.total_tokens for r in test_results if r.token_usage),
    total_estimated_cost=sum(r.estimated_cost for r in test_results if r.estimated_cost is not None),
    notes=[
        "IRS test slice is very small (~4-5 documents); IRS per-class metrics are "
        "statistically noisy and should be reported as uncertain, not precise."
    ],
)
save_metrics_report(test_report, settings)
test_report.model_dump()

{'method': 'llm',
 'split': 'test',
 'accuracy': 0.9867549668874173,
 'macro_precision': 0.9535256410256411,
 'macro_recall': 0.9892045454545455,
 'macro_f1': 0.9693874078630312,
 'weighted_f1': 0.9870158453278095,
 'per_class': [{'label': 'USCIS',
   'precision': 0.9807692307692307,
   'recall': 1.0,
   'f1': 0.9902912621359223,
   'support': 51},
  {'label': 'DMV',
   'precision': 1.0,
   'recall': 0.9818181818181818,
   'f1': 0.9908256880733946,
   'support': 55},
  {'label': 'SSA',
   'precision': 1.0,
   'recall': 0.975,
   'f1': 0.9873417721518988,
   'support': 40},
  {'label': 'IRS',
   'precision': 0.8333333333333334,
   'recall': 1.0,
   'f1': 0.9090909090909091,
   'support': 5}],
 'confusion_matrix': [[51, 0, 0, 0],
  [1, 54, 0, 0],
  [0, 0, 39, 1],
  [0, 0, 0, 5]],
 'confusion_matrix_labels': ['USCIS', 'DMV', 'SSA', 'IRS'],
 'mean_latency_ms': 1507.9145834377246,
 'total_token_usage': 1041448,
 'total_estimated_cost': 0.1098985,
 'cost_per_document': 0.0007278046357615894,

## Summary for the next notebook

- LLM validation macro F1 (for demo default-routing selection): see `val_report.macro_f1`.
- LLM frozen test macro F1 (for the research comparison): see `test_report.macro_f1`.
- Row-level predictions and metrics are saved to `artifacts/predictions/` and
  `artifacts/reports/` for both splits.
- Next: `07_rag_index_creation.ipynb` builds the routing-only RAG index from `train.csv`.